In [1]:
import pandas as pd
import numpy as np
from collections import defaultdict, Counter

In [2]:
# bước chuẩn bị dữ liệu
%run Prepare_data.ipynb

Note: you may need to restart the kernel to use updated packages.


<class 'pandas.DataFrame'>
RangeIndex: 5220 entries, 0 to 5219
Data columns (total 24 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   rating             5220 non-null   int64         
 1   title_x            5220 non-null   str           
 2   text               5220 non-null   str           
 3   images_x           5220 non-null   object        
 4   asin               5220 non-null   str           
 5   parent_asin        5220 non-null   str           
 6   user_id            5220 non-null   str           
 7   timestamp          5220 non-null   datetime64[ns]
 8   helpful_vote       5220 non-null   int64         
 9   verified_purchase  5220 non-null   bool          
 10  main_category      5220 non-null   str           
 11  title_y            5220 non-null   str           
 12  average_rating     5220 non-null   float64       
 13  rating_number      5220 non-null   int64         
 14  features           

# Các hàm xây dựng phương pháp Popular-Based

In [21]:
def precision_at_k(recommended, actual, k=10):
    if not recommended:
        return 0.0
    recommended_k = recommended[:k]
    actual_set = set(actual)
    hits = len(set(recommended_k) & actual_set)
    return hits / k

def recall_at_k(recommended, actual, k=10):
    if not actual:
        return 0.0
    recommended_k = recommended[:k]
    actual_set = set(actual)
    hits = len(set(recommended_k) & actual_set)
    return hits / len(actual_set)

def ndcg_at_k(recommended, actual, k=10):
    if not recommended:
        return 0.0
    recommended_k = recommended[:k]
    actual_set = set(actual)
    dcg = 0.0
    for i, item in enumerate(recommended_k):
        if item in actual_set:
            dcg += 1.0 / np.log2(i + 2)
    ideal_hits = min(len(actual_set), k)
    idcg = sum(1.0 / np.log2(i + 2) for i in range(ideal_hits))
    return dcg / idcg if idcg > 0 else 0.0

In [22]:
def build_user_ground_truth(test_df):
    """
    Xây dựng ground truth cho bài toán user-to-item.
    Parameters:
        test_df: DataFrame chứa 'user_id', 'parent_asin', 'liked'
    Returns:
        dict: user_id -> set of parent_asin (các item liked trong test)
    """
    user_gt = test_df[test_df['liked'] == 1].groupby('user_id')['parent_asin'].apply(set).to_dict()
    return user_gt

def build_item_ground_truth(train_df, top_m=20):
    """
    Xây dựng ground truth cho bài toán item-to-item dựa trên co-occurrence.
    Parameters:
        train_df: DataFrame chứa 'user_id', 'parent_asin', 'liked' (chỉ lấy liked=1)
        top_m: số lượng item liên quan tối đa cho mỗi item (dùng để giới hạn ground truth)
    Returns:
        dict: parent_asin -> set of parent_asin (các item liên quan)
    """
    # Chỉ xét các tương tác liked=1
    liked_df = train_df[train_df['liked'] == 1]
    # Nhóm các item theo user
    user_items = liked_df.groupby('user_id')['parent_asin'].apply(set).to_dict()
    
    # Đếm số lần hai item cùng xuất hiện
    cooccur = defaultdict(Counter)
    for items in user_items.values():
        items_list = list(items)
        for i in range(len(items_list)):
            for j in range(i+1, len(items_list)):
                a, b = items_list[i], items_list[j]
                cooccur[a][b] += 1
                cooccur[b][a] += 1
    
    # Với mỗi item, lấy top_m item có số lần cùng xuất hiện cao nhất
    item_gt = {}
    for item, counter in cooccur.items():
        # Lấy top_m item (có thể ít hơn nếu không đủ)
        top_items = [it for it, _ in counter.most_common(top_m)]
        item_gt[item] = set(top_items)
    
    return item_gt

In [23]:
def compute_popular_items(train_df, method='weighted', m=None):
    """
    Tính điểm phổ biến cho từng sản phẩm dựa trên tập train.
    
    Parameters:
        train_df (pd.DataFrame): DataFrame có cột 'parent_asin' và 'liked'
        method (str): 'count' hoặc 'weighted'
        m (float): ngưỡng cho weighted method (nếu None, dùng median của số lượt tương tác)
    
    Returns:
        list: Danh sách parent_asin đã sắp xếp giảm dần theo điểm phổ biến
    """
    if method == 'count':
        # Điểm = tổng số lượt liked
        scores = train_df.groupby('parent_asin')['liked'].sum()
        scores = scores.sort_values(ascending=False)
    elif method == 'weighted':
        # Thống kê cho mỗi sản phẩm
        stats = train_df.groupby('parent_asin')['liked'].agg(
            liked_sum='sum',
            liked_mean='mean',
            count='count'
        )
        C = train_df['liked'].mean()   # tỷ lệ liked toàn bộ
        if m is None:
            m = stats['count'].median()  # median của số lượt tương tác
        # Công thức IMDB: (v/(v+m))*R + (m/(v+m))*C
        stats['score'] = (stats['count'] / (stats['count'] + m)) * stats['liked_mean'] \
                         + (m / (stats['count'] + m)) * C
        stats = stats.sort_values('score', ascending=False)
        scores = stats['score']
    else:
        raise ValueError("method must be 'count' or 'weighted'")
    
    return scores.index.tolist()

In [24]:
def popular_recommend_user(user_id, interacted_items, popular_items, k=10):
    """
    Gợi ý top-k sản phẩm phổ biến nhất, loại bỏ các sản phẩm đã tương tác.
    
    Parameters:
        user_id (str): không dùng nhưng giữ để đồng bộ với các mô hình khác
        interacted_items (set): set các parent_asin user đã tương tác (train)
        popular_items (list): danh sách sản phẩm phổ biến đã sắp xếp
        k (int): số lượng gợi ý
    
    Returns:
        list: top-k parent_asin
    """
    recs = [item for item in popular_items if item not in interacted_items]
    return recs[:k]

In [25]:
def popular_recommend_item(item_id, popular_items, k=10):
    """
    Gợi ý top-k sản phẩm phổ biến nhất, loại bỏ chính item_id nếu có.
    (Lưu ý: popular không dựa vào item gốc, chỉ gợi ý chung)
    """
    recs = [it for it in popular_items if it != item_id]
    return recs[:k]

In [26]:
def evaluate_user_to_item(train_df, test_df, recommend_func, k=10):
    """
    Đánh giá mô hình user-to-item.
    """
    # Sử dụng hàm ground truth đã định nghĩa
    user_actual = build_user_ground_truth(test_df)  # thay vì viết trực tiếp
    user_interacted = train_df.groupby('user_id')['parent_asin'].apply(set).to_dict()
    
    precisions = []
    recalls = []
    ndcgs = []
    
    for user, actual in user_actual.items():
        if not actual:
            continue
        interacted = user_interacted.get(user, set())
        recommended = recommend_func(user, interacted, k)
        recommended = [item for item in recommended if item not in interacted]
        precisions.append(precision_at_k(recommended, actual, k))
        recalls.append(recall_at_k(recommended, actual, k))
        ndcgs.append(ndcg_at_k(recommended, actual, k))
    
    return {
        'precision': np.mean(precisions),
        'recall': np.mean(recalls),
        'ndcg': np.mean(ndcgs)
    }

In [27]:
def evaluate_item_to_item(train_df, item_ground_truth, recommend_func_item, k=10):
    """
    Đánh giá mô hình item-to-item.
    
    Parameters:
        train_df: không dùng trực tiếp, nhưng giữ để đồng bộ
        item_ground_truth: dict {item: set of related items}
        recommend_func_item: hàm func(item_id, k) -> list
        k: số lượng gợi ý
    """
    precisions = []
    recalls = []
    ndcgs = []
    
    for item, actual in item_ground_truth.items():
        if not actual:
            continue
        recommended = recommend_func_item(item, k)
        p = precision_at_k(recommended, actual, k)
        r = recall_at_k(recommended, actual, k)
        n = ndcg_at_k(recommended, actual, k)
        precisions.append(p)
        recalls.append(r)
        ndcgs.append(n)
    
    return {
        'precision': np.mean(precisions),
        'recall': np.mean(recalls),
        'ndcg': np.mean(ndcgs)
    }

# Thu kết quả đánh giá từ phương pháp Popular-Based

In [28]:
# đã có train_df, test_df

# 1. Tính popular items
popular_items = compute_popular_items(train_df, method='weighted', m=None)
print(f"Số lượng popular items: {len(popular_items)}")

# 2. Đánh giá user-to-item
def recommend_user(user_id, interacted, k=10):
    return popular_recommend_user(user_id, interacted, popular_items, k)

user_results = evaluate_user_to_item(train_df, test_df, recommend_user, k=10)

print("\n=== POPULAR MODEL (USER-TO-ITEM) ===")
print(f"Precision@10: {user_results['precision']:.4f}")
print(f"Recall@10   : {user_results['recall']:.4f}")
print(f"NDCG@10     : {user_results['ndcg']:.4f}")

# 3. Đánh giá item-to-item
item_gt = build_item_ground_truth(train_df, top_m=20)
def recommend_item(item_id, k=10):
    return popular_recommend_item(item_id, popular_items, k)

item_results = evaluate_item_to_item(train_df, item_gt, recommend_item, k=10)

print("\n=== POPULAR MODEL (ITEM-TO-ITEM) ===")
print(f"Precision@10: {item_results['precision']:.4f}")
print(f"Recall@10   : {item_results['recall']:.4f}")
print(f"NDCG@10     : {item_results['ndcg']:.4f}")

Số lượng popular items: 2148

=== POPULAR MODEL (USER-TO-ITEM) ===
Precision@10: 0.0013
Recall@10   : 0.0100
NDCG@10     : 0.0087

=== POPULAR MODEL (ITEM-TO-ITEM) ===
Precision@10: 0.0277
Recall@10   : 0.0226
NDCG@10     : 0.0286
